<a href="https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_2B_Thinking_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Qwen3.5-2B in thinking mode

Step 2 of 2. Trains Qwen3.5-2B on the checked reasoning examples from [step 1](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_9B_Generate_Reasoning_Examples.ipynb), so it
reasons, searches the docs and checks its GDScript the way Qwen3.5-9B did.

- **The training data is mostly reasoning examples.** Unsloth advises keeping at least 75% reasoning examples to preserve thinking mode. The rest (20% by default) are non-thinking docs examples, as in the [first notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_2B_Godot_Docs_Tool_Calling.ipynb), so non-thinking mode keeps working.
- **It's measured before and after training** on held-out tasks the 9B never solved: does the 2B finish thinking (Qwen notes that the 2B is more prone to thinking loops than larger Qwen3.5 models), search the docs, check its code, and write GDScript that passes Godot's checker?

**Runtime:** `Runtime > Change runtime type > T4 GPU` (free tier) is enough.

## 1. Install Unsloth

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Settings

In [ ]:
HF_DATASET = "YOUR_HF_USERNAME/godot-reasoning-examples"  # made by the generation notebook
NON_THINKING_SHARE = 0.2  # share of non-thinking docs examples; Unsloth advises at least 75% reasoning examples
MAX_LENGTH = 8192         # same as MAX_EXAMPLE_TOKENS in the generation notebook
NUM_EVAL_TASKS = 10       # held-out tasks to measure before and after training (each can take a few minutes)
MAX_NEW_TOKENS = 4096     # per reply when evaluating; a reply that hits it counts as never finishing

## 3. Tools

The same tools and system prompt as in step 1, so the 2B is trained on exactly what it will use.

In [ ]:
import hashlib, os, pathlib, subprocess, urllib.request, zipfile

GODOT_VERSION = "4.7.2-stable"
GODOT_DIR = pathlib.Path("godot").resolve()
GODOT = str(GODOT_DIR / f"Godot_v{GODOT_VERSION}_linux.x86_64")

if not os.path.exists(GODOT):
    release = f"https://github.com/godotengine/godot/releases/download/{GODOT_VERSION}"
    archive = f"Godot_v{GODOT_VERSION}_linux.x86_64.zip"
    zip_path, _ = urllib.request.urlretrieve(f"{release}/{archive}")
    sums = urllib.request.urlopen(f"{release}/SHA512-SUMS.txt").read().decode()
    expected = next(line.split()[0] for line in sums.splitlines() if line.endswith(archive))
    actual = hashlib.sha512(pathlib.Path(zip_path).read_bytes()).hexdigest()
    assert actual == expected, "Godot download doesn't match the official SHA-512 checksum"
    zipfile.ZipFile(zip_path).extractall(GODOT_DIR)
    os.chmod(GODOT, 0o755)
print(subprocess.run([GODOT, "--version"], capture_output = True, text = True).stdout)

In [ ]:
import pathlib, re, subprocess, tempfile

SCRIPT_ERROR = re.compile(r"SCRIPT ERROR: (?:Parse Error: )?(.*)\n\s*at: .*?:(\d+)\)")

def check_gdscript(code: str) -> str:
    """Checks a GDScript file with Godot's own parser (no code is run) and returns the errors."""
    with tempfile.TemporaryDirectory() as project:  # a separate project per check, so checks can run in parallel
        (pathlib.Path(project) / "project.godot").write_text("config_version=5\n")
        (pathlib.Path(project) / "check.gd").write_text(code, encoding = "utf-8")
        try:
            result = subprocess.run(
                [GODOT, "--headless", "--path", project, "--check-only", "--script", "res://check.gd"],
                stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, timeout = 60)
        except subprocess.TimeoutExpired:
            return "The check timed out."
    errors = [f"Line {line}: {message}" for message, line in SCRIPT_ERROR.findall(result.stdout)]
    if errors:
        return "Errors:\n" + "\n".join(errors)
    if result.returncode != 0:  # failed without the usual error format: show Godot's output
        return "Errors:\n" + result.stdout[-1500:]
    return "No errors found."

print(check_gdscript("extends Node2D\n\nfunc _ready() -> void:\n    velocity = Vector2.ZERO\n"))

In [ ]:
from datasets import load_dataset, Features, Value

columns = ["instruction", "input", "output", "category", "source"]
docs = load_dataset("arunsrajan/godot-docs-dataset", split = "train", keep_default_na = False,
                    features = Features({column: Value("string") for column in columns})).to_list()
print(len(docs), "rows")

In [ ]:
import math, re
from collections import Counter, defaultdict

STOPWORDS = {"the", "a", "an", "of", "in", "on", "to", "and", "or", "is", "are", "do", "does",
             "how", "what", "when", "why", "which", "i", "my", "me", "it", "for", "with", "godot",
             "explain", "about", "tell", "this", "that", "can", "you"}

def tokenize(text):
    words = [w for w in re.findall(r"[a-z0-9_]+", text.lower()) if w not in STOPWORDS]
    # "look_at" is also indexed as "look" and "at", so "look at" finds it.
    return words + [part for w in words if "_" in w for part in w.split("_") if part and part not in STOPWORDS]

class BM25:
    """Small keyword search index (BM25 ranking), no extra packages needed."""
    def __init__(self, texts, k1 = 1.5, b = 0.75):
        self.k1, self.b = k1, b
        self.postings = defaultdict(list)  # term -> [(doc_id, term_count)]
        self.lengths = []
        for doc_id, text in enumerate(texts):
            counts = Counter(tokenize(text))
            self.lengths.append(sum(counts.values()))
            for term, count in counts.items():
                self.postings[term].append((doc_id, count))
        self.avg_length = sum(self.lengths) / len(self.lengths)
        n = len(texts)
        self.idf = {term: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5)) for term, p in self.postings.items()}

    def search(self, query, k = 3):
        scores = defaultdict(float)
        for term in set(tokenize(query)):
            for doc_id, count in self.postings.get(term, ()):
                norm = count + self.k1 * (1 - self.b + self.b * self.lengths[doc_id] / self.avg_length)
                scores[doc_id] += self.idf[term] * count * (self.k1 + 1) / norm
        return sorted(scores, key = scores.get, reverse = True)[:k]

In [ ]:
index = BM25([row["instruction"] + "\n" + row["output"] for row in docs])

In [ ]:
MAX_RESULT_CHARS = 1500
DOCS_URL = "https://docs.godotengine.org/en/latest/"

def format_results(doc_ids):
    parts = []
    for rank, doc_id in enumerate(doc_ids, 1):
        row = docs[doc_id]
        text = row["output"]
        if len(text) > MAX_RESULT_CHARS:
            text = text[:MAX_RESULT_CHARS] + " ..."
        parts.append(f"[{rank}] {DOCS_URL}{row['source']}\n{text}")
    return "\n\n".join(parts) if parts else "No results."

def search_godot_docs(query: str) -> str:
    return format_results(index.search(query, k = 3))

TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_godot_docs",
        "description": "Search the official Godot Engine 4.7 documentation (class reference, tutorials and manuals). "
                       "Returns the 3 most relevant passages with their URLs.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "What to look up, e.g. 'Node2D look_at' or 'detect collisions CharacterBody2D'."},
            },
            "required": ["query"],
        },
    },
}]

In [ ]:
TOOLS.append({
    "type": "function",
    "function": {
        "name": "check_gdscript",
        "description": "Checks a complete GDScript file with the Godot 4 engine's own parser and returns the parse and "
                       "type errors with line numbers, or 'No errors found.'. It does not run the code, so it cannot "
                       "find logic errors. It cannot check C#.",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string",
                         "description": "The complete GDScript file, starting with its extends line."},
            },
            "required": ["code"],
        },
    },
})
TOOL_FUNCTIONS = {"search_godot_docs": search_godot_docs, "check_gdscript": check_gdscript}

The system prompt is the same as in the [coding assistant notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_Godot_Coding_Assistant.ipynb).

In [ ]:
SYSTEM_PROMPT = (
    "You are a Godot Engine 4 coding assistant for GDScript and C#.\n"
    "- Use search_godot_docs to look up any Godot class, method, property or signal you are not sure about. "
    "Use only APIs that exist in Godot 4; never invent them.\n"
    "- Write GDScript with static types, for example `func _physics_process(delta: float) -> void:`, "
    "so the checker can find type errors.\n"
    "- Before answering with GDScript, call check_gdscript on the complete script and fix every error it reports. "
    "It only finds parse and type errors, so also check the logic yourself.\n"
    "- check_gdscript cannot check C#. For C#, follow the C# examples in the search results.\n"
    "- Give the final code in one fenced code block, then explain it briefly."
)

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*<function=([\w.-]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
PARAMETER = re.compile(r"<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def parse_tool_calls(reply):
    """Reads Qwen3.5's '<tool_call><function=name><parameter=x>...' blocks from a model reply."""
    calls = [{"name": name, "arguments": {k: v.strip() for k, v in PARAMETER.findall(body)}}
             for name, body in TOOL_CALL.findall(reply)]
    return calls, TOOL_CALL.sub("", reply).strip()


GDSCRIPT_BLOCK = re.compile(r"```(?:gdscript|gd)\n(.*?)```", re.DOTALL)

def run_tool(call):
    function = TOOL_FUNCTIONS.get(call["name"])
    if function is None:
        return f"Unknown tool: {call['name']}"
    try:
        return function(**call["arguments"])
    except TypeError as error:
        return f"Wrong arguments for {call['name']}: {error}"

## 4. Load Qwen3.5-2B

16-bit LoRA, as Unsloth recommends for Qwen3.5. LoRA adapters start at zero, so until training the model behaves
exactly like the base model.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-2B",
    max_seq_length = MAX_LENGTH,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
# Qwen3.5 is a vision-language model; for text we only need the text tokenizer.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Training data

The reasoning dataset is private, so log in with a Hugging Face token that can read it.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import json
from datasets import load_dataset

reasoning = load_dataset(HF_DATASET)
reasoning_conversations = [json.loads(messages) for messages in reasoning["train"]["messages"]]
EVAL_TASKS = reasoning["test"]["task"][:NUM_EVAL_TASKS]
print(len(reasoning_conversations), "reasoning examples |", len(EVAL_TASKS), "evaluation tasks")

Non-thinking examples are built as in the first notebook (question → search → docs passage with its source), from
docs passages the reasoning examples don't use. They render with an empty `<think></think>` block, which is how
Qwen3.5's non-thinking mode looks.

In [ ]:
import random

def build_conversation(doc_id, rng):
    row = docs[doc_id]
    question = row["instruction"]
    hits = index.search(question, k = 3)
    if doc_id not in hits:
        hits = hits[:2] + [doc_id]  # make sure the right passage is among the results
    rng.shuffle(hits)              # ... but not always in the same position
    answer = row["output"] + f"\n\nSource: {DOCS_URL}{row['source']}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": "search_godot_docs", "arguments": {"query": question}}}]},
        {"role": "tool", "content": format_results(hits)},
        {"role": "assistant", "content": answer},
    ]

In [ ]:
import random
from datasets import Dataset

rng = random.Random(3407)
num_plain = round(len(reasoning_conversations) * NON_THINKING_SHARE / (1 - NON_THINKING_SHARE))
used_sources = set(reasoning["train"]["source"]) | set(reasoning["test"]["source"])
candidates = [i for i, row in enumerate(docs) if "(part " not in row["instruction"]
              and len(row["output"]) <= MAX_RESULT_CHARS and row["source"] not in used_sources]
plain_conversations = [build_conversation(i, rng) for i in rng.sample(candidates, num_plain)]

texts, dropped = [], 0
for conversation in reasoning_conversations + plain_conversations:
    text = text_tokenizer.apply_chat_template(conversation, tools = TOOLS, tokenize = False)
    if len(text_tokenizer(text, add_special_tokens = False)["input_ids"]) <= MAX_LENGTH:
        texts.append(text)
    else:
        dropped += 1
train_dataset = Dataset.from_dict({"text": texts}).shuffle(seed = 3407)
print(f"{len(reasoning_conversations)} reasoning + {num_plain} non-thinking examples "
      f"({len(reasoning_conversations) / (len(reasoning_conversations) + num_plain):.0%} reasoning), {dropped} too long")

In [ ]:
print(next(text for text in train_dataset["text"] if "<think>\n\n</think>" not in text)[:4000])

## 6. Measure before training

Each task runs the full tool loop in thinking mode. With `MAX_NEW_TOKENS = 4096` per reply, a reply that never
closes its `<think>` block counts as not finishing. This can take several minutes per task on a T4.

In [ ]:
import collections, time
from transformers import TextStreamer

def generate(messages, thinking = True, stream = False):
    prompt = text_tokenizer.apply_chat_template(messages, tools = TOOLS, add_generation_prompt = True,
                                                tokenize = False, enable_thinking = thinking)
    inputs = text_tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to(model.device)
    sampling = dict(temperature = 0.6, top_p = 0.95, top_k = 20) if thinking else dict(temperature = 0.7, top_p = 0.8, top_k = 20)
    streamer = TextStreamer(text_tokenizer, skip_prompt = True, skip_special_tokens = True) if stream else None
    output = model.generate(**inputs, max_new_tokens = MAX_NEW_TOKENS, do_sample = True, streamer = streamer, **sampling)
    return text_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def run_conversation(task, thinking = True, max_rounds = 6, verbose = False):
    """The tool loop. Returns (answer, tools used, failure); failure is None when the model answered."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]
    tools_used = []
    for _ in range(max_rounds):
        reply = generate(messages, thinking, stream = verbose)
        if thinking and "</think>" not in reply:
            return None, tools_used, "never finished thinking"
        reasoning_text, rest = reply.split("</think>", 1) if "</think>" in reply else ("", reply)
        calls, content = parse_tool_calls(rest.strip())
        message = {"role": "assistant", "content": content, "reasoning_content": reasoning_text.replace("<think>", "").strip()}
        if not calls:
            return content, tools_used, None
        message["tool_calls"] = [{"type": "function", "function": call} for call in calls]
        messages.append(message)
        for call in calls:
            result = run_tool(call)
            tools_used.append(call["name"])
            messages.append({"role": "tool", "content": result})
            if verbose:
                print(f"\n----- [tool] {call['name']} -> {result[:500]}\n")
    return None, tools_used, "too many tool calls"

def evaluate(tasks, label):
    """Thinking-mode results on held-out tasks: does it finish, use the tools, and write GDScript that passes?"""
    stats, started = collections.Counter(), time.time()
    for task in tasks:
        answer, tools_used, failure = run_conversation(task)
        blocks = GDSCRIPT_BLOCK.findall(answer or "")
        stats["finished (no loop)"] += failure is None
        stats["searched the docs"] += "search_godot_docs" in tools_used
        stats["checked its code"] += "check_gdscript" in tools_used
        stats["final GDScript passes"] += bool(blocks) and all(check_gdscript(code) == "No errors found." for code in blocks)
    print(f"{label}: {len(tasks)} tasks in {time.time() - started:.0f}s")
    for name in ["finished (no loop)", "searched the docs", "checked its code", "final GDScript passes"]:
        print(f"  {name:22s} {stats[name]}/{len(tasks)}")
    return stats

In [ ]:
FastLanguageModel.for_inference(model)
before = evaluate(EVAL_TASKS, "Before training")

## 7. Train

`train_on_responses_only` trains on the assistant turns only, including their reasoning, and not on the system
prompt, the tasks or the tool results.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model = model,
    tokenizer = text_tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Check the masking: only the assistant turns, with their reasoning, should remain (masked tokens show as spaces).

In [ ]:
labels = trainer.train_dataset[0]["labels"]
space = text_tokenizer(" ", add_special_tokens = False).input_ids[0]
print(text_tokenizer.decode([space if token == -100 else token for token in labels])[:3000])

In [ ]:
trainer_stats = trainer.train()

## 8. Measure after training

In [ ]:
FastLanguageModel.for_inference(model)
after = evaluate(EVAL_TASKS, "After training")
print(f"\n{'':22s} before  after")
for name in ["finished (no loop)", "searched the docs", "checked its code", "final GDScript passes"]:
    print(f"{name:22s} {before[name]:>3}/{len(EVAL_TASKS)}  {after[name]:>3}/{len(EVAL_TASKS)}")

Watch one task, streaming the thinking and tool calls:

In [ ]:
answer, tools_used, failure = run_conversation(EVAL_TASKS[0], verbose = True)
print("=" * 80, "\n" + (answer or failure))

## 9. Save

Run the model in thinking mode with the same tools and system prompt, for example in the
[coding assistant notebook](https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_Godot_Coding_Assistant.ipynb) with `MODEL_NAME` set to your saved model and `THINKING = True`.
Replace `YOUR_HF_USERNAME` / `YOUR_HF_TOKEN` to push to the Hugging Face Hub.

In [ ]:
model.save_pretrained("qwen3_5_2b_godot_thinking_lora")
text_tokenizer.save_pretrained("qwen3_5_2b_godot_thinking_lora")

if False: model.push_to_hub_merged("YOUR_HF_USERNAME/qwen3.5-2b-godot-thinking", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")
if False: model.push_to_hub_gguf("YOUR_HF_USERNAME/qwen3.5-2b-godot-thinking-GGUF", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")